In [1]:
import math
from IPython.core.pylabtools import figsize
import joblib
import pandas as pd
import numpy as np
from glob import glob
import os
import matplotlib.pyplot as plt
from joblib import Parallel, delayed



cell_names = [cell[23:] for cell in list(pd.read_csv("/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C1/cell_ratio.csv").columns)[1:]]

case = "23508"
slide = "23508_D2"

save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/super_resolution/stnet/inference/"+slide+"_2x_resolution.pkl"
pred_and_label = joblib.load(save_path)

pred_and_label[slide]['cell_abundance_predictions'] = np.clip(pred_and_label[slide]['cell_abundance_predictions'], a_min=0, a_max=None)

cell2location_counts = pred_and_label[slide]['cell_abundance_labels']

label_mask = np.sum(cell2location_counts, axis=1) >= 0
label_mask_index = np.nonzero(label_mask)[0]
cell2location_counts = cell2location_counts[label_mask_index]

hist2cell2x_counts = pred_and_label[slide]['cell_abundance_predictions']

coordinates = pred_and_label[slide]['coords']
cell2location_coordinates = coordinates[label_mask_index]
hist2cell2x_coordinates = coordinates

In [19]:
os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/super_resolution/stnet/all_in_one", slide))

# Calculate the minimum distance between points
min_distance = np.min([math.sqrt((coordinates[i][0] - coordinates[j][0])**2 + (coordinates[i][1] - coordinates[j][1])**2) for i in range(len(coordinates)) for j in range(i+1, len(coordinates))])
# Set the node size as a fraction of the minimum distance (e.g., 25%)
node_size = min_distance * 0.15

for cell_id in range(39):
    figsize(11, 5)

    # Create subplots for cell2location_count2, hist2cell_counts, and stnet_counts
    fig, axs = plt.subplots(1, 2)
    # Reduce the space between subplots
    plt.subplots_adjust(wspace=0.02)

    i = 0

    # Set the background color of each scatter plot to light gray
    axs[0].set_facecolor('lightgray')
    axs[1].set_facecolor('lightgray')

    # cell2location_count2 plot
    scatter_plot1 = axs[0].scatter(cell2location_coordinates[:, 0], cell2location_coordinates[:, 1], c=cell2location_counts[:, cell_id], cmap='magma', s=node_size*4)
    axs[0].invert_yaxis()
    min_coord = min(min(cell2location_coordinates[:, 0]), min(cell2location_coordinates[:, 1]))
    max_coord = max(max(cell2location_coordinates[:, 0]), max(cell2location_coordinates[:, 1]))
    # axs[0].set_xlim(min_coord-300, max_coord+300)
    # axs[0].set_ylim(max_coord+300, min_coord-300)
    axs[0].set_xticks([])
    axs[0].set_yticks([])
    axs[0].set_title("Cell2location")
    cbar1 = fig.colorbar(scatter_plot1, ax=axs[0], shrink=0.69, aspect=28)

    # hist2cell_counts plot
    scatter_plot2 = axs[1].scatter(hist2cell2x_coordinates[:, 0], hist2cell2x_coordinates[:, 1], c=hist2cell2x_counts[:, cell_id], cmap='magma', s=node_size)
    axs[1].invert_yaxis()
    min_coord = min(min(hist2cell2x_coordinates[:, 0]), min(hist2cell2x_coordinates[:, 1]))
    max_coord = max(max(hist2cell2x_coordinates[:, 0]), max(hist2cell2x_coordinates[:, 1]))
    # axs[1].set_xlim(min_coord-300, max_coord+300)
    # axs[1].set_ylim(max_coord+300, min_coord-300)
    axs[1].set_xticks([])
    axs[1].set_yticks([])
    axs[1].set_title("Hist2Cell_2x")
    cbar2 = fig.colorbar(scatter_plot2, ax=axs[1], shrink=0.69, aspect=28)

    # # Set the aspect ratio of both subplots to be equal
    axs[0].set_aspect('equal', adjustable='box')
    axs[1].set_aspect('equal', adjustable='box')
    

    # Save the figure without white margins
    plt.savefig(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/super_resolution/stnet/all_in_one", slide, cell_names[cell_id]+".png"), bbox_inches='tight')
    # plt.show()
    plt.close()
    # break